In [1]:
# NORTHSTAR — Solace Browser: Settings + Dashboard + Tunnel QA
# DNA: page_qa(settings,dashboard,tunnel) = structure(HTML) x sections(coverage) x a11y(WCAG) x i18n(keys) x security(CSP)
# Pages: settings.html, machine-dashboard.html, tunnel-connect.html
# Towers: T1(Masters) T6(Hackers) T8(Elders) T9(World) T23(Web)
# Auth: 65537 | File-based probes — reads HTML directly (no running server)
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "settings-dashboard-tunnel-qa"
NOTEBOOK_PATH = "notebooks/qa/28-settings-dashboard-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
LOCALES_DIR = BROWSER / "app" / "locales" / "yinyang"

TARGET_PAGES = ["settings", "machine-dashboard", "tunnel-connect"]

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

def read_page(name):
    return (WEB / f"{name}.html").read_text(encoding="utf-8")

# Verify all target pages exist
for page in TARGET_PAGES:
    assert (WEB / f"{page}.html").exists(), f"MISSING: {page}.html"

print(f"BOOTSTRAP: {len(TARGET_PAGES)} target pages")
print(f"Pages: {TARGET_PAGES}")
print(f"Locales dir exists: {LOCALES_DIR.exists()}")

BOOTSTRAP: 3 target pages
Pages: ['settings', 'machine-dashboard', 'tunnel-connect']
Locales dir exists: True


In [2]:
# P1: Structure — each page has proper HTML5 structure, loads theme.js + layout.js
print("=== P1: HTML5 Structure + Required Scripts ===")

REQUIRED_STRUCTURE = {
    "doctype": r"<!DOCTYPE html>",
    "html_lang": r'<html\s+lang="[a-z]{2}"',
    "meta_charset": r'<meta\s+charset="[uU][tT][fF]-?8"',
    "title_tag": r"<title>.+</title>",
}

REQUIRED_SCRIPTS = ["theme.js", "layout.js"]

for page in TARGET_PAGES:
    html = read_page(page)

    # HTML5 structure checks
    for check_name, pattern in REQUIRED_STRUCTURE.items():
        found = bool(re.search(pattern, html, re.IGNORECASE))
        record(f"P1-{page}-{check_name}", f"{page}.html has {check_name}", found,
               detail=f"Pattern {pattern!r} not found" if not found else "",
               tower_ref="T1,T23")

    # Script loading checks
    for script in REQUIRED_SCRIPTS:
        found = bool(re.search(rf'<script\s[^>]*src="[^"]*{re.escape(script)}"', html))
        record(f"P1-{page}-{script}", f"{page}.html loads {script}", found,
               detail=f"{script} not loaded via <script> tag" if not found else "",
               tower_ref="T1,T23")

passed = sum(1 for r in results if r["id"].startswith("P1") and r["status"] == "PASS")
total = sum(1 for r in results if r["id"].startswith("P1"))
print(f"\nP1 complete: {passed}/{total} passed")

=== P1: HTML5 Structure + Required Scripts ===
  PASS: settings.html has doctype
  PASS: settings.html has html_lang
  PASS: settings.html has meta_charset
  PASS: settings.html has title_tag
  PASS: settings.html loads theme.js
  PASS: settings.html loads layout.js
  PASS: machine-dashboard.html has doctype
  PASS: machine-dashboard.html has html_lang
  PASS: machine-dashboard.html has meta_charset
  PASS: machine-dashboard.html has title_tag
  PASS: machine-dashboard.html loads theme.js
  PASS: machine-dashboard.html loads layout.js
  PASS: tunnel-connect.html has doctype
  PASS: tunnel-connect.html has html_lang
  PASS: tunnel-connect.html has meta_charset
  PASS: tunnel-connect.html has title_tag
  PASS: tunnel-connect.html loads theme.js
  PASS: tunnel-connect.html loads layout.js

P1 complete: 18/18 passed


In [3]:
# P2: Settings — appearance (theme), API keys, tier display, accessibility options
print("=== P2: Settings Page Sections ===")

settings_html = read_page("settings")

# Appearance / theme section
has_appearance = bool(re.search(r'id="appearance"|[Aa]ppearance|theme-grid|data-theme-choice', settings_html))
record("P2-appearance", "settings.html has Appearance/theme section", has_appearance,
       detail="No appearance or theme section found" if not has_appearance else "",
       tower_ref="T1,T8")

theme_choices = re.findall(r'data-theme-choice="([^"]+)"', settings_html)
record("P2-theme-choices", f"settings.html offers theme choices: {theme_choices}",
       len(theme_choices) >= 2,
       detail=f"Expected >=2 theme choices, found {len(theme_choices)}" if len(theme_choices) < 2 else "",
       tower_ref="T1,T8")

# API keys section
has_api_keys = bool(re.search(r'[Aa][Pp][Ii]\s*[Kk]ey|api.?key|authentication|auth.*status', settings_html, re.IGNORECASE))
record("P2-api-keys", "settings.html has API key / authentication section", has_api_keys,
       detail="No API key or authentication section found" if not has_api_keys else "",
       tower_ref="T1,T6")

# Tier display
has_tier = bool(re.search(r'tier|plan|free|starter|pro\b', settings_html, re.IGNORECASE))
record("P2-tier-display", "settings.html mentions user tier/plan", has_tier,
       detail="No tier/plan display found" if not has_tier else "",
       tower_ref="T1,T2")

# Accessibility options (text size, reduced motion, high contrast)
has_accessibility = bool(re.search(r'text.?size|font.?size|accessibility|reduced.?motion|high.?contrast', settings_html, re.IGNORECASE))
record("P2-accessibility", "settings.html has accessibility options", has_accessibility,
       detail="No accessibility options found" if not has_accessibility else "",
       tower_ref="T1,T8")

passed = sum(1 for r in results if r["id"].startswith("P2") and r["status"] == "PASS")
total = sum(1 for r in results if r["id"].startswith("P2"))
print(f"\nP2 complete: {passed}/{total} passed")

=== P2: Settings Page Sections ===
  PASS: settings.html has Appearance/theme section
  PASS: settings.html offers theme choices: ['dark', 'light', 'midnight']
  PASS: settings.html has API key / authentication section
  PASS: settings.html mentions user tier/plan
  PASS: settings.html has accessibility options

P2 complete: 5/5 passed


In [4]:
# P3: Dashboard — machine info, terminal preview, status indicators
print("=== P3: Machine Dashboard Page ===")

dash_html = read_page("machine-dashboard")

# Machine / system info section
has_machine_info = bool(re.search(r'[Ss]ystem\s*(posture|info|snapshot)|machine.?state|system-table', dash_html, re.IGNORECASE))
record("P3-machine-info", "machine-dashboard.html has system/machine info section", has_machine_info,
       detail="No system info or machine state section found" if not has_machine_info else "",
       tower_ref="T1,T23")

# Terminal preview
has_terminal = bool(re.search(r'terminal-output|terminal-form|command-input|terminal.?preview|[Cc]ommand\s*preview', dash_html))
record("P3-terminal", "machine-dashboard.html has terminal preview", has_terminal,
       detail="No terminal preview element found" if not has_terminal else "",
       tower_ref="T1,T23")

# Status indicators (status-tag classes or similar)
status_tags = re.findall(r'status-tag--(\w+)', dash_html)
record("P3-status-indicators", f"machine-dashboard.html has status indicators: {set(status_tags)}",
       len(status_tags) >= 1,
       detail="No status-tag indicators found" if not status_tags else "",
       tower_ref="T1,T23")

# Loads theme.js + layout.js (already checked in P1 but verify dashboard specifically)
has_layout = bool(re.search(r'layout\.js', dash_html))
has_theme = bool(re.search(r'theme\.js', dash_html))
record("P3-scripts", "machine-dashboard.html loads theme.js + layout.js",
       has_layout and has_theme,
       detail=f"theme.js={has_theme}, layout.js={has_layout}" if not (has_layout and has_theme) else "",
       tower_ref="T1,T23")

passed = sum(1 for r in results if r["id"].startswith("P3") and r["status"] == "PASS")
total = sum(1 for r in results if r["id"].startswith("P3"))
print(f"\nP3 complete: {passed}/{total} passed")

=== P3: Machine Dashboard Page ===
  PASS: machine-dashboard.html has system/machine info section
  PASS: machine-dashboard.html has terminal preview
  PASS: machine-dashboard.html has status indicators: {'success', 'warning'}
  PASS: machine-dashboard.html loads theme.js + layout.js

P3 complete: 4/4 passed


In [5]:
# P4: Tunnel — consent flow, connection status, CORS/security mentions
print("=== P4: Tunnel Connect Page ===")

tunnel_html = read_page("tunnel-connect")

# Consent flow — connect/disconnect buttons, approval language
has_consent = bool(re.search(r'connect-button|disconnect-button|[Rr]equest\s*tunnel|[Cc]lose\s*tunnel|[Gg]rant\s*tunnel', tunnel_html))
record("P4-consent-flow", "tunnel-connect.html has consent flow (connect/disconnect)", has_consent,
       detail="No consent flow elements found (connect/disconnect buttons)" if not has_consent else "",
       tower_ref="T1,T6")

# Connection status display
has_status = bool(re.search(r'status-ring|status-label|[Dd]isconnected|[Cc]onnecting|[Cc]onnected|endpoint-value', tunnel_html))
record("P4-connection-status", "tunnel-connect.html has connection status display", has_status,
       detail="No connection status elements found" if not has_status else "",
       tower_ref="T1,T23")

# Security mentions (CSP, CORS, or general security language)
has_csp = bool(re.search(r'Content-Security-Policy', tunnel_html))
record("P4-csp", "tunnel-connect.html has CSP meta tag", has_csp,
       detail="No Content-Security-Policy meta tag" if not has_csp else "",
       tower_ref="T6")

# OAuth3 or scope language (tunnel.connect scope)
has_scope = bool(re.search(r'tunnel\.connect|oauth3|scope|permission|approval', tunnel_html, re.IGNORECASE))
record("P4-scope-mention", "tunnel-connect.html mentions scope/permission/approval", has_scope,
       detail="No tunnel scope or permission language found" if not has_scope else "",
       tower_ref="T1,T6")

# Endpoint copy/open buttons
has_endpoint_actions = bool(re.search(r'copy-endpoint|open-endpoint|[Cc]opy\s*URL|[Oo]pen\s*URL', tunnel_html))
record("P4-endpoint-actions", "tunnel-connect.html has endpoint copy/open actions", has_endpoint_actions,
       detail="No endpoint action buttons found" if not has_endpoint_actions else "",
       tower_ref="T1,T23")

passed = sum(1 for r in results if r["id"].startswith("P4") and r["status"] == "PASS")
total = sum(1 for r in results if r["id"].startswith("P4"))
print(f"\nP4 complete: {passed}/{total} passed")

=== P4: Tunnel Connect Page ===
  PASS: tunnel-connect.html has consent flow (connect/disconnect)
  PASS: tunnel-connect.html has connection status display
  PASS: tunnel-connect.html has CSP meta tag
  PASS: tunnel-connect.html mentions scope/permission/approval
  PASS: tunnel-connect.html has endpoint copy/open actions

P4 complete: 5/5 passed


In [6]:
# P5: Evidence summary — aggregate results, compute score, seal hash
print("=== P5: Evidence Summary ===")

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = round(100 * passed / total, 1) if total > 0 else 0.0

print(f"\n{'='*60}")
print(f"  NOTEBOOK: {NOTEBOOK_PATH}")
print(f"  PROJECT:  {PROJECT}")
print(f"  PAGES:    {TARGET_PAGES}")
print(f"  TOTAL:    {total} probes")
print(f"  PASSED:   {passed}")
print(f"  FAILED:   {failed}")
print(f"  SCORE:    {score}%  (min required: {MIN_SCORE}%)")
print(f"{'='*60}")

if failed > 0:
    print("\nFAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['id']} — {r['name']}")
            if r["detail"]:
                print(f"        {r['detail']}")

# Seal
evidence = {
    "notebook": NOTEBOOK_PATH,
    "northstar": NORTHSTAR,
    "project": PROJECT,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "total": total,
    "passed": passed,
    "failed": failed,
    "score_pct": score,
    "min_score": MIN_SCORE,
    "verdict": "PASS" if score >= MIN_SCORE else "FAIL",
    "results": results,
}

seal = hashlib.sha256(json.dumps(evidence, sort_keys=True).encode()).hexdigest()[:16]
evidence["seal"] = seal

print(f"\n  VERDICT: {evidence['verdict']}")
print(f"  SEAL:    {seal}")
print(f"  AUTH:    65537")

=== P5: Evidence Summary ===

  NOTEBOOK: notebooks/qa/28-settings-dashboard-qa.ipynb
  PROJECT:  solace-browser
  PAGES:    ['settings', 'machine-dashboard', 'tunnel-connect']
  TOTAL:    32 probes
  PASSED:   32
  FAILED:   0
  SCORE:    100.0%  (min required: 70%)

  VERDICT: PASS
  SEAL:    24cdac0cb4ca6e28
  AUTH:    65537
